In [5]:
!pip install datasets
from datasets import load_dataset
import pandas as pd

raw_dataset = load_dataset("wangrongsheng/ag_news")

df= pd.DataFrame(raw_dataset['train'])
print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [8]:
import re

# Function to clean text
def clean_text(text):
    text = text.lower()                      # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove numbers and special characters
    text = re.sub(r'\s+', ' ', text)         # Remove extra spaces
    return text.strip()

# Apply cleaning
df['text'] = df['text'].apply(clean_text)

# Display cleaned data
print(df[['text', 'label']].head())

                                                text  label
0  wall st bears claw back into the black reuters...      2
1  carlyle looks toward commercial aerospace reut...      2
2  oil and economy cloud stocks outlook reuters r...      2
3  iraq halts oil exports from main southern pipe...      2
4  oil prices soar to alltime record posing new m...      2


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer()

# Build vocabulary
tokenizer.fit_on_texts(df['text'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['text'])

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)
print("First Sequence:", sequences[0])

Vocabulary Size: 91344
First Sequence: [391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295]


In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Shape of X:", X.shape)
print(X[0])

Shape of X: (120000, 50)
[  391   324  1525 14260    99    54     1   812    23    23 38863   391
  1988 50537     4 38864    34  3893   737   295     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [11]:
from tensorflow.keras.utils import to_categorical

# Convert labels
y = to_categorical(df['label'])

print("Shape of y:", y.shape)
print(y[:5])

Shape of y: (120000, 4)
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(96000, 50)
(24000, 50)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(4, activation="softmax")
])

# Build the model
model.build(input_shape=(None, max_length))

# Display model architecture
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 128)        │    11,692,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,733,444 (44.76 MB)

 Trainable params: 11,733,444 (44.76 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 205s 168ms/step - accuracy: 0.7821 - loss: 0.6054 - val_accuracy: 0.8438 - val_loss: 0.4869
Epoch 2/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 202s 168ms/step - accuracy: 0.8928 - loss: 0.3406 - val_accuracy: 0.8442 - val_loss: 0.4985
Epoch 3/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 207s 172ms/step - accuracy: 0.9224 - loss: 0.2488 - val_accuracy: 0.8528 - val_loss: 0.4862
Epoch 4/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 206s 172ms/step - accuracy: 0.9319 - loss: 0.2202 - val_accuracy: 0.8403 - val_loss: 0.4893
Epoch 5/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 205s 171ms/step - accuracy: 0.9416 - loss: 0.1853 - val_accuracy: 0.7664 - val_loss: 0.7638


In [16]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7690 - loss: 0.7748
Test Loss: 0.7748204469680786
Test Accuracy: 0.7690416574478149
